# Detector de piezas v2 — DESDE CERO (dataset limpio)

Este notebook entrena **(a) el detector YOLO** de la arquitectura v2 con el
dataset reconstruido desde la posición inicial física correcta.

| | |
|---|---|
| Modelo base | `yolo11n.pt` (COCO, `MODO = "desde_cero"`) |
| Paquete | `detector_v2_clean.zip` (~287 MB) en `Mi unidad/ajedrez_v2/` |
| Train | 4,044 tableros generados: 1,544 sintéticos + 2,500 compuestos limpios |
| Valid | 109 recortes **reales** (01/02/03/05, sin repetir posición) |
| Test | 110 recortes **reales** (02/03/04/05 + parte de 01) |
| Épocas | 120, `batch=16`, `imgsz=640`, `lr0=0.01` |

**Por qué desde cero y no desde `modelo_sintetico.pt`:** ese peso se entrenó con
la convención vieja de fichas y los compuestos heredaron el cruce caballo/alfil.
Un peso contaminado contamina el reentreno: aquí se parte de COCO.

**La regla que no se toca:** `valid` y `test` son SOLO fotos reales de la cámara.
El train es todo lo generado. Si val o test tuvieran imágenes sintéticas, la
métrica mediría lo que el modelo ya vio y no serviría para decidir nada.

Antes de ejecutar: **Entorno de ejecución > Cambiar tipo de entorno > T4 GPU**.

## 1. Verificar la GPU

In [ ]:
#@title 1. Verificar GPU y versiones
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("SIN GPU: ve a Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU")

## 2. Dependencias

Colab ya trae `torch` y casi siempre `ultralytics`. **No se reinstala nada por
gusto**: instalar `ultralytics` a mano arrastra otra versión de `torch` y puede
romper la GPU de la sesión. La celda sólo instala si falta el paquete.

In [ ]:
#@title 2. Comprobar ultralytics sin tocar torch
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3,<9"],
        check=True,
    )

import ultralytics
print("ultralytics", ultralytics.__version__)
ultralytics.checks()

## 3. Traer el paquete de datos

Sube `detector_v2_clean.zip` a `Mi unidad/ajedrez_v2/` y deja `USAR_DRIVE = True`.
El zip se descomprime en `/content` (disco local de la VM), **no** en Drive:
leer miles de imágenes montadas desde Drive es mucho más lento.

In [ ]:
#@title 3. Descomprimir detector_v2_clean.zip y contar por split
import shutil
import zipfile
from pathlib import Path

USAR_DRIVE = True
ZIP_EN_DRIVE = Path("/content/drive/MyDrive/ajedrez_v2/detector_v2_clean.zip")
DESTINO = Path("/content/ajedrez_v2/detector")
ESPERADO = {"train": 4372, "val": 73, "test": 64}

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    origen = ZIP_EN_DRIVE
else:
    from google.colab import files
    subidos = files.upload()
    origen = Path(next(iter(subidos)))

if not origen.exists():
    raise FileNotFoundError(
        f"No encuentro {origen}. Sube detector_v2_clean.zip a "
        "Mi unidad/ajedrez_v2/ o pon USAR_DRIVE = False."
    )

if DESTINO.exists():
    shutil.rmtree(DESTINO)
DESTINO.mkdir(parents=True)

with zipfile.ZipFile(origen) as zf:
    zf.extractall(DESTINO)

print("Contenido:", sorted(p.name for p in DESTINO.iterdir()))
for split, esperado in ESPERADO.items():
    n_img = len(list((DESTINO / "images" / split).glob("*.jpg")))
    n_lbl = len(list((DESTINO / "labels" / split).glob("*.txt")))
    estado = "OK" if n_img == esperado == n_lbl else "REVISAR"
    print(f"  {split:5s} imagenes={n_img:5d} etiquetas={n_lbl:5d} "
          f"(esperado {esperado:5d}) {estado}")

assert (DESTINO / "data.yaml").exists(), "Falta data.yaml en el zip"

PROYECTO = Path("/content/drive/MyDrive/ajedrez_v2/detector")
PROYECTO.mkdir(parents=True, exist_ok=True)
print("Registro de entrenamientos:", PROYECTO)

In [ ]:
#@title 3b. Corregir `path` del data.yaml (REQUERIDO por Ultralytics)
# Sin esto, Ultralytics resuelve `path` contra su datasets_dir GLOBAL
# (/content/datasets) y falla con: "missing path '/content/images/val'".
import re

RUTA_YAML = DESTINO / "data.yaml"
texto = RUTA_YAML.read_text(encoding="utf-8")
texto, cambios = re.subn(r"^path: .*$", f"path: {DESTINO.as_posix()}",
                         texto, flags=re.M)
RUTA_YAML.write_text(texto, encoding="utf-8")
print(f"path reescrito en {RUTA_YAML} ({cambios} cambio(s))")
assert cambios >= 1, "data.yaml no tenia linea 'path:'"
print("\n".join(texto.splitlines()[:6]))

## 4. Auditar el dataset antes de entrenar

Atrapa los errores silenciosos: imagen y etiqueta con el mismo nombre en los
tres splits, líneas YOLO válidas con clases en rango, `data.yaml` con las 12
clases en el orden de `classes.txt` y ninguna imagen repetida entre splits.

In [ ]:
#@title 4. Auditoria del dataset
from collections import Counter

import yaml

CLASES_ESPERADAS = [
    "peon_b", "torre_b", "caballo_b", "alfil_b", "dama_b", "rey_b",
    "peon_n", "torre_n", "caballo_n", "alfil_n", "dama_n", "rey_n",
]
RAIZ = DESTINO
cfg = yaml.safe_load((RAIZ / "data.yaml").read_text(encoding="utf-8"))
NOMBRES = [cfg["names"][i] for i in range(cfg["nc"])]
print("Suites:", cfg["train"], "|", cfg["val"], "|", cfg["test"])
print("Clases:", NOMBRES)
if NOMBRES != CLASES_ESPERADAS:
    raise SystemExit("data.yaml no respeta el orden de classes.txt: NO entrenes asi")

problemas = []
resumen = {}
vistos = {}

for split in ("train", "val", "test"):
    dir_img = RAIZ / "images" / split
    dir_lbl = RAIZ / "labels" / split
    imgs = {p.stem for p in dir_img.glob("*.jpg")}
    lbls = {p.stem for p in dir_lbl.glob("*.txt")}
    if imgs - lbls:
        problemas.append(f"{split}: {len(imgs - lbls)} imagenes sin etiqueta")
    if lbls - imgs:
        problemas.append(f"{split}: {len(lbls - imgs)} etiquetas sin imagen")

    conteo = Counter()
    vacias = 0
    invalidas = 0
    for txt in dir_lbl.glob("*.txt"):
        lineas = [l for l in txt.read_text(encoding="utf-8").splitlines() if l.strip()]
        if not lineas:
            vacias += 1
        for linea in lineas:
            partes = linea.split()
            if len(partes) != 5:
                invalidas += 1
                continue
            clase = int(partes[0])
            x, y, w, h = (float(v) for v in partes[1:])
            if not (0 <= clase < 12) or not all(0.0 < v <= 1.0 for v in (x, y, w, h)):
                invalidas += 1
                continue
            conteo[clase] += 1
    vistos[split] = imgs
    resumen[split] = {"imagenes": len(imgs), "cajas": sum(conteo.values()),
                      "conteo": conteo}
    print(f"{split:5s} imagenes={len(imgs):5d} cajas={sum(conteo.values()):6d} "
          f"negativos={vacias:3d} invalidas={invalidas}")
    if invalidas:
        problemas.append(f"{split}: {invalidas} lineas invalidas")

for a in ("train", "val", "test"):
    for b in ("train", "val", "test"):
        if a < b and vistos[a] & vistos[b]:
            problemas.append(f"{len(vistos[a] & vistos[b])} imagenes repetidas "
                             f"entre {a} y {b}")

for split in ("val", "test"):
    faltan = [NOMBRES[i] for i in range(12) if resumen[split]["conteo"][i] == 0]
    if faltan:
        print(f"AVISO {split}: clases sin ejemplos -> {faltan}")

print()
if problemas:
    print("PROBLEMAS:")
    for p in problemas:
        print("  -", p)
    raise SystemExit("Corrige el dataset antes de entrenar.")
print("Dataset OK. Puedes entrenar.")

## 5. Configuración del entrenamiento

`MODO = "desde_cero"`: se parte de **`yolo11n.pt`** (pesos COCO) y se entrena
todo el modelo con `lr0 = 0.01`. Desde cero necesita más learning rate que el
fine-tune anterior (0.001) porque no hay nada que preservar.

Aumentos pensados para un tablero, no para sprites:

- `fliplr = 0.0`, `flipud = 0.0`: el tablero tiene orientación **fija** y de ella
  depende el orden de clases (a1 abajo-izquierda). Voltear convierte una torre
  blanca en una negra de otra esquina.
- `mosaic = 0.5` y `close_mosaic = 15`: mosaico suave y las últimas 15 épocas con
  imágenes completas.
- `cache = "ram"`: 4,044 imágenes caben en la RAM de Colab.
- `seed = 42`, `deterministic = True`: misma corrida, mismo resultado.

El proyecto se escribe en **Drive** para no perder `best.pt` si la sesión cae.

In [ ]:
#@title 5. Configuracion
MODO = "desde_cero"     # "desde_cero" (yolo11n.pt) | "fino" (reanudar un .pt)
BASE = "yolo11n.pt"     # MODO="desde_cero": pesos COCO, NO el modelo viejo
EPOCAS = 120
BATCH = 16              # YOLO11n a 640 px entra comodo en una T4
IMGSZ = 640
LR0 = 0.01              # desde cero; el fine-tune usaba 0.001
PATIENCE = 25
NOMBRE_RUN = "desde_cero_v2_clean"

DATA = DESTINO / "data.yaml"
assert DATA.exists(), "Falta el paquete: vuelve a la celda 3"

print("Modo:", MODO)
print("Base:", BASE)
print("Epocas:", EPOCAS, "| batch:", BATCH, "| imgsz:", IMGSZ, "| lr0:", LR0)
print("Salida:", PROYECTO / NOMBRE_RUN)

In [ ]:
#@title 6. Entrenar
from pathlib import Path

from ultralytics import YOLO

COMUNES = dict(
    data=str(DATA),
    imgsz=IMGSZ,
    batch=BATCH,
    workers=2,
    cache="ram",
    seed=42,
    deterministic=True,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.5,
    close_mosaic=15,
    cos_lr=True,
    patience=PATIENCE,
    plots=True,
    val=True,
    project=str(PROYECTO),
    name=NOMBRE_RUN,
    exist_ok=True,
    save_period=5,
)

modelo = YOLO(BASE if MODO == "desde_cero" else MODO)
resultado = modelo.train(epochs=EPOCAS, lr0=LR0, lrf=0.01, **COMUNES)

PESO_FINAL = Path(resultado.save_dir) / "weights" / "best.pt"
print("Modelo final:", PESO_FINAL, "| existe:", PESO_FINAL.exists())

## 6. Evaluar sobre `test` (fotos reales, nunca vistas)

Este es el número que vale. `valid` decide cuándo parar (lo usa `patience`);
`test` es la medida final.

**No hay línea base comparable.** El modelo del lote anterior daba `mAP50 0.546`
pero se entrenó con etiquetas cruzadas caballo/alfil y sobre otro test: ese
número **no** es una referencia honesta para este dataset. La comparación válida
es contra la primera época de esta misma corrida (está en `results.csv`).

In [ ]:
#@title 7. Evaluar en test + metricas por clase
from ultralytics import YOLO

modelo_eval = YOLO(str(PESO_FINAL))
metricas = modelo_eval.val(data=str(DATA), split="test", imgsz=IMGSZ,
                           batch=BATCH, plots=True, project=str(PROYECTO),
                           name="eval_test", exist_ok=True)

print()
print(f"TEST  mAP50   : {metricas.box.map50:.4f}")
print(f"TEST  mAP50-95: {metricas.box.map:.4f}")
print(f"TEST  precision: {metricas.box.mp:.4f}")
print(f"TEST  recall   : {metricas.box.mr:.4f}")

print()
print(f"{'clase':12s} {'P':>7s} {'R':>7s} {'mAP50-95':>9s}")
for i, nombre in metricas.names.items():
    p = metricas.box.p[i] if i < len(metricas.box.p) else 0.0
    r = metricas.box.r[i] if i < len(metricas.box.r) else 0.0
    m = metricas.box.maps[i] if i < len(metricas.box.maps) else 0.0
    print(f"{nombre:12s} {p:7.3f} {r:7.3f} {m:9.3f}")

## 7. Confusiones reales -> predichas

La matriz de confusión dice qué clases se confunden entre sí. El par a vigilar
es `caballo_n <-> alfil_n`: fue el que se cruzó en las etiquetas del lote
anterior. Si aquí aparece con fuerza, el problema es de datos, no de épocas.

In [ ]:
#@title 8. Confusiones mas frecuentes
import numpy as np

cm = getattr(getattr(metricas, "confusion_matrix", None), "matrix", None)
if cm is None:
    print("Esta version de Ultralytics no expuso la matriz de confusion.")
else:
    cm = np.asarray(cm)
    nombres = metricas.names
    pares = []
    for real in range(min(len(nombres), cm.shape[0])):
        for pred in range(min(len(nombres), cm.shape[1])):
            if real != pred and cm[real, pred] > 0:
                pares.append((float(cm[real, pred]), nombres[real], nombres[pred]))
    pares.sort(reverse=True)
    print("Confusiones mas frecuentes (real -> predicha):")
    if not pares:
        print("  Ninguna. Excelente.")
    for cantidad, real, pred in pares[:12]:
        print(f"  {real:12s} -> {pred:12s} {cantidad:g}")

## 8. Guardar todo en Drive

Sin esta celda, si Colab cierra la sesión se pierde el entrenamiento. `last.pt`
permite reanudar; `best.pt` es el modelo que usa el pipeline.

In [ ]:
#@title 9. Copiar modelo y curvas a Drive
import shutil

SALIDA = PROYECTO / "entrega"
SALIDA.mkdir(parents=True, exist_ok=True)

shutil.copy2(PESO_FINAL, SALIDA / "best.pt")

ultimo = PESO_FINAL.parent / "last.pt"
if ultimo.exists():
    shutil.copy2(ultimo, SALIDA / "last.pt")

run_dir = PESO_FINAL.parent.parent
for patron in ("results.csv", "results.png", "confusion_matrix.png",
               "confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png"):
    for archivo in run_dir.glob(patron):
        shutil.copy2(archivo, SALIDA / archivo.name)

shutil.copy2(DATA, SALIDA / "data.yaml")

print("Guardado en", SALIDA)
for p in sorted(SALIDA.iterdir()):
    print(f"  {p.name:38s} {p.stat().st_size / 1024:8.1f} KB")

## 9. Prueba visual rápida

In [ ]:
#@title 10. Predecir sobre una foto de test
import random

from IPython.display import display
from PIL import Image

fotos = sorted((DESTINO / "images" / "test").glob("*.jpg"))
elegida = random.choice(fotos)
print("Foto:", elegida.name)

prediccion = modelo_eval.predict(str(elegida), imgsz=IMGSZ, conf=0.40, verbose=False)[0]
salida = Path("/content/prediccion_v2_clean.jpg")
Image.fromarray(prediccion.plot()[..., ::-1]).save(salida)
display(Image.open(salida))
print(f"{len(prediccion.boxes)} detecciones")

## Si algo falla

| Síntoma | Causa | Solución |
|---|---|---|
| `SIN GPU` | Entorno sin GPU | Entorno de ejecución > Cambiar tipo de entorno > T4 GPU |
| `No encuentro ...detector_v2_clean.zip` | Ruta de Drive distinta | Revisa `ZIP_EN_DRIVE` o pon `USAR_DRIVE = False` |
| `data.yaml no respeta el orden` | Zip editado a mano | Vuelve a generarlo con `scripts/preparar_colab_v2_clean.py` |
| `CUDA out of memory` | `BATCH` alto | Baja `BATCH` a 8 |
| mAP50 alto en valid y bajo en test | Sobreajuste a val | Baja `EPOCAS`, sube `PATIENCE` |
| Se perdió el entrenamiento | Colab cerró la sesión | Reanuda con `YOLO(...last.pt).train(resume=True)` |